In [ ]:
"""
this notebook detects the bacterial cells that are "peripheral" ("loose" - not part of the "aggregate")
and removes them.
this is to make the radius of gyration estimation be more accurate
without this, the presence of loose cells can inflate the radii inconsistently leading to wrong estimates. 
this notebook also allows viz of everything that is kept and everyting that is removed



"""

In [ ]:


# just peripheral cells viz

file_path = "Image 1_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv"

import pandas as pd
import numpy as np
import plotly.graph_objects as go

def analyze_peripheral_z_span(csv_path, skiprows=1, distance_percentile=90):
    """
    Identifies peripheral cells based on XY distance from centroid,
    computes their Z span, and visualizes in Plotly.
    
    Parameters:
    - csv_path: Path to CSV file.
    - skiprows: Number of rows to skip (default 1).
    - distance_percentile: Percentile to define peripheral cells (default 90).
    
    Returns:
    - periph_z_min, periph_z_max, z_span: Peripheral Z span details.
    - df: DataFrame with computed distances and peripheral flags.
    """
    df = pd.read_csv(csv_path, skiprows=skiprows)
    
    # Compute XY centroid
    centroid_x = df['Cube center coord_x (vox)'].mean()
    centroid_y = df['Cube center coord_y (vox)'].mean()
    
    # Compute XY distance from centroid
    df['dist_xy'] = np.sqrt((df['Cube center coord_x (vox)'] - centroid_x)**2 +
                            (df['Cube center coord_y (vox)'] - centroid_y)**2)
    
    # Define peripheral cells based on distance threshold
    distance_threshold = np.percentile(df['dist_xy'], distance_percentile)
    df['peripheral'] = df['dist_xy'] > distance_threshold
    
    # Extract Z values of peripheral cells
    periph_z_min = df[df['peripheral']]['Cube center coord_z (vox)'].min()
    periph_z_max = df[df['peripheral']]['Cube center coord_z (vox)'].max()
    z_span = periph_z_max - periph_z_min
    
    print(f"Peripheral Z span: min={periph_z_min}, max={periph_z_max}, span={z_span}")
    
    return periph_z_min, periph_z_max, z_span, df

def plot_peripheral_cells_plotly(df, title='Peripheral Cells in 3D'):
    """
    Visualizes the data in 3D Plotly scatter, highlighting peripheral cells.
    
    Parameters:
    - df: DataFrame with 'Cube center coord_x (vox)', 'Cube center coord_y (vox)',
          'Cube center coord_z (vox)', 'peripheral'.
    - title: Plot title.
    """
    # Peripheral and core points
    periph_df = df[df['peripheral']]
    core_df = df[~df['peripheral']]
    
    trace_core = go.Scatter3d(
        x=core_df['Cube center coord_x (vox)'],
        y=core_df['Cube center coord_y (vox)'],
        z=core_df['Cube center coord_z (vox)'],
        mode='markers',
        marker=dict(size=3, color='blue', opacity=0.6),
        name='Core Cells'
    )
    trace_periph = go.Scatter3d(
        x=periph_df['Cube center coord_x (vox)'],
        y=periph_df['Cube center coord_y (vox)'],
        z=periph_df['Cube center coord_z (vox)'],
        mode='markers',
        marker=dict(size=3, color='red', opacity=0.8),
        name='Peripheral Cells'
    )
    
    fig = go.Figure(data=[trace_core, trace_periph])
    fig.update_layout(
        scene=dict(
            xaxis_title='X (vox)',
            yaxis_title='Y (vox)',
            zaxis_title='Z (vox)'
        ),
        title=title
    )
    fig.show()

#  usage
csv_path = file_path
periph_z_min, periph_z_max, z_span, df_with_flags = analyze_peripheral_z_span(csv_path)
plot_peripheral_cells_plotly(df_with_flags)

# final code that

### reads a list of csv files, finds peripheral cells to determine what is the width of one layer of cells, removes that much from the bottom and saves a new csv file

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

def analyze_peripheral_z_span(csv_path, skiprows=1, distance_percentile=90):
    df = pd.read_csv(csv_path, skiprows=skiprows)
    centroid_x = df['Cube center coord_x (vox)'].mean()
    centroid_y = df['Cube center coord_y (vox)'].mean()
    df['dist_xy'] = np.sqrt((df['Cube center coord_x (vox)'] - centroid_x)**2 +
                            (df['Cube center coord_y (vox)'] - centroid_y)**2)
    distance_threshold = np.percentile(df['dist_xy'], distance_percentile)
    df['peripheral'] = df['dist_xy'] > distance_threshold
    periph_z_min = df[df['peripheral']]['Cube center coord_z (vox)'].min()
    periph_z_max = df[df['peripheral']]['Cube center coord_z (vox)'].max()
    z_span = periph_z_max - periph_z_min
    print(f"File: {csv_path}")
    print(f"Peripheral Z span: min={periph_z_min}, max={periph_z_max}, span={z_span}")
    return periph_z_min, periph_z_max, z_span, df

def filter_and_visualize_and_save(csv_path, skiprows=1, distance_percentile=90, layer_thickness_vox=1):
    periph_z_min, periph_z_max, z_span, df = analyze_peripheral_z_span(csv_path, skiprows, distance_percentile)
    z_cutoff = periph_z_min + layer_thickness_vox
    print(f"Removing Z layers from {periph_z_min} to {z_cutoff} (thickness = {layer_thickness_vox} voxels)")

    removed_df = df[df['Cube center coord_z (vox)'] <= z_cutoff]
    kept_df = df[df['Cube center coord_z (vox)'] > z_cutoff]

    # Visualization
    trace_removed = go.Scatter3d(
        x=removed_df['Cube center coord_x (vox)'],
        y=removed_df['Cube center coord_y (vox)'],
        z=removed_df['Cube center coord_z (vox)'],
        mode='markers',
        marker=dict(size=3, color='gray', opacity=0.3),
        name='Removed Layer'
    )
    trace_kept = go.Scatter3d(
        x=kept_df['Cube center coord_x (vox)'],
        y=kept_df['Cube center coord_y (vox)'],
        z=kept_df['Cube center coord_z (vox)'],
        mode='markers',
        marker=dict(
            size=3,
            color=kept_df['Cube center coord_z (vox)'],
            colorscale='Viridis',
            opacity=0.8
        ),
        name='Kept Data'
    )

    fig = go.Figure(data=[trace_removed, trace_kept])
    fig.update_layout(
        scene=dict(
            xaxis_title='X (vox)',
            yaxis_title='Y (vox)',
            zaxis_title='Z (vox)'
        ),
        title=f'Filtered Data: {os.path.basename(csv_path)}'
    )
    fig.show()

    # Save filtered data
    base, ext = os.path.splitext(csv_path)
    output_file = f"{base}_filtered_for_loose_cells.csv"
    kept_df.drop(columns=['dist_xy', 'peripheral'], errors='ignore').to_csv(output_file, index=False)
    print(f"Filtered CSV saved as: {output_file}")

# List of CSV files
csv_files =['/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 6_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 1_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 2_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 3_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 4_5isto1_032725_pos01_t01_filtered_roi.ome_colony1._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 4_5isto1_032725_pos01_t01_filtered_roi.ome_colony2._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 4_5isto1_032725_pos01_t01_filtered_roi.ome_colony3._pos1_ch2_frame000001.csv','/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/ww density analysis/removal of z slices from green ch2 data/Image 5_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001.csv']

# Process each file
for csv_path in csv_files:
    try:
        filter_and_visualize_and_save(csv_path)
    except Exception as e:
        print(f"Error processing {csv_path}: {e}")

In [ ]:

# simple viz function. 
# in case you want to visualize any one of the iamges without any gray areas

csv_path = 'Image 1_5isto1_032725_pos01_t01_filtered_roi.ome._pos1_ch2_frame000001_filtered_for_loose_cells.csv'

import pandas as pd
import plotly.graph_objects as go

# Load the filtered CSV
df = pd.read_csv(csv_path)

# Plot in Plotly
fig = go.Figure(data=[go.Scatter3d(
    x=df['Cube center coord_x (vox)'],
    y=df['Cube center coord_y (vox)'],
    z=df['Cube center coord_z (vox)'],
    mode='markers',
    marker=dict(
        size=3,
        color=df['Cube center coord_z (vox)'],
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(title='Z')
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title='X (vox)',
        yaxis_title='Y (vox)',
        zaxis_title='Z (vox)'
    ),
    title=f'Filtered Data Visualization: {csv_path}'
)

fig.show()